In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, median_absolute_error

In [ ]:
df_train = pd.read_csv('../data/processed/task6/train_cleaned6.csv')
df_val = pd.read_csv('../data/processed/task6/val_cleaned6.csv')

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)

print("\nTrain columns:")
print(df_train.columns.tolist())

Train shape: (5048, 70)
Validation shape: (1249, 70)

Train columns:
['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'is_phone_listing', 'lg_model', 'model_key', 'match_rule', 'confidence', 'family', 'storage_known', 'ram_known', 'condition_known', 'storage_imp_src', 'storage_imp_conf', 'storage_imp_post', 'ram_imp_src', 'ram_imp_conf', 'ram_imp_post', 'imp_price_used', 'ref_matched', 'storage_gb_orig', 'storage_gb_was_imputed', 'ram_gb_orig', 'ram_gb_was_imputed', 'ref_n', 'ref_med_log', 'ref_mad_log', 'ref_level', 'ref_median_price', 'price

In [ ]:
features = [
    'model_key', 'brand', 'family', 'storage_gb', 'ram_gb',
    'condition', 'has_5g', 'has_warranty', 'has_box',
    'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint',
    'is_tablet_hint', 'is_parts_hint', 'accessory_first',
    'bundle_hint', 'is_feature_phone_hint', 'loc_area',
    'loc_city'
]

cat_features = [
    'model_key', 'brand', 'family',
    'condition', 'loc_area', 'loc_city'
]

catboost_params = {
    'iterations': 1500,
    'learning_rate': 0.03,
    'depth': 6,
    'eval_metric': 'MAE',
    'random_seed': 42,
    'verbose': 0,
    'allow_writing_files': False
}

required_columns = (features + ['price', 'title_group', 'listing_type'])

for name, data in [('train', df_train), ('validation', df_val)]:
    missing = sorted(set(required_columns) - set(data.columns))
    if missing:
        raise ValueError(f"Missing columns in {name}: {missing}")

In [ ]:
def prepare(data):
    X = data[features].copy()
    X[cat_features] = (X[cat_features].fillna('Unknown').astype(str))
    return X

train_model_data = df_train[df_train['price'].notna()].copy()

# Keep only phone listings with valid prices for evaluation
val_phones = df_val[(df_val['listing_type'] == 'phone') &
    df_val['price'].notna() &
    df_val['title_group'].notna()
].copy()

# half for early stopping, half for evaluation
splitter = GroupShuffleSplit(n_splits=1,test_size=0.50,random_state=42) 

early_idx, eval_idx = next(splitter.split(val_phones,groups=val_phones['title_group']))
early_stop = val_phones.iloc[early_idx].copy()
eval_df = val_phones.iloc[eval_idx].copy()



In [10]:

model = CatBoostRegressor(loss_function='Quantile:alpha=0.5',**catboost_params)

model.fit(prepare(train_model_data),train_model_data['price'],cat_features=cat_features,
    eval_set=(
        prepare(early_stop),
        early_stop['price']
    ),
    early_stopping_rounds=100, verbose=False
)

eval_df['predicted_price'] = model.predict(
    prepare(eval_df)
)

eval_df['absolute_error'] = (
    eval_df['price'] - eval_df['predicted_price']
).abs()

print("Training ads:", len(train_model_data))
print("Early stopping ads:", len(early_stop))
print("Evaluation phone ads:", len(eval_df))
print("Best iteration:", model.best_iteration_)
print(
    "Overall MAE:",
    round(mean_absolute_error(
        eval_df['price'],
        eval_df['predicted_price']
    ), 2)
)

Training ads: 5048
Early stopping ads: 658
Evaluation phone ads: 494
Best iteration: 1485
Overall MAE: 6251.13


In [11]:
eval_df

,title,price,location,ad_url,ad_id,run_id,brand,storage_gb,ram_gb,storage_src,...,price_flag,price_context,listing_type,listing_type_rule,dup_group_size,dup_candidate_id,dup_key_strength,drop_by_rule,predicted_price,absolute_error
2,سامسونج اس 24 الترا,40000.0,الهضبة الوسطي، المقطم,/ad/%D8%B3%D8%A7%D9%85%D8%B3%D9%88%D9%86%D8%AC...,208958079,legacy,samsung,256.0,12.0,NaN,...,False,NaN,phone,phone_brand_detected,1,NaN,NaN,False,39305.236797,694.763203
4,Iphone 13 pro max,37000.0,اطلالة، العباسية,/ad/iphone-13-pro-max-ID207445230.html,207445230,legacy,apple,256.0,6.0,NaN,...,False,NaN,phone,phone_brand_detected,1,NaN,NaN,False,30680.139143,6319.860857
5,سامسونج اس 24 الترا,45000.0,الزقازيق، الشرقية,/ad/%D8%B3%D8%A7%D9%85%D8%B3%D9%88%D9%86%D8%AC...,208968556,legacy,samsung,256.0,12.0,NaN,...,False,NaN,phone,phone_brand_detected,1,NaN,NaN,False,38294.093693,6705.906307
9,سامسونج اس 25 الترا,54000.0,المطرية، القاهرة,/ad/%D8%B3%D8%A7%D9%85%D8%B3%D9%88%D9%86%D8%AC...,209012798,legacy,samsung,256.0,12.0,NaN,...,False,NaN,phone,phone_brand_detected,2,val_dup_0001,weak,False,54375.879471,375.879471
10,شاومي ريدمي نوت 11 تى برو+,8000.0,بولاق الدكرور، الجيزة,/ad/%D8%B4%D8%A7%D9%88%D9%85%D9%8A-%D8%B1%D9%8...,209012736,legacy,xiaomi,128.0,8.0,NaN,...,False,bundle,phone,phone_brand_detected,1,NaN,NaN,False,6347.788292,1652.211708
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1233,Apple - iPhone 16 Pro Max 512 — pristine condi...,75000.0,التجمع الاول، القاهرة الجديدة,/ad/apple-iphone-16-pro-max-512-pristine-condi...,208944099,legacy,apple,512.0,8.0,L1:bare,...,False,NaN,phone,phone_brand_detected,1,NaN,NaN,False,73359.762329,1640.237671
1234,ايفون I phone Plus 7,9000.0,بنها، القليوبية,/ad/%D8%A7%D9%8A%D9%81%D9%88%D9%86-i-phone-plu...,208944044,legacy,apple,32.0,3.0,NaN,...,True,NaN,phone,phone_brand_detected,1,NaN,NaN,False,3692.669143,5307.330857
1236,Apple - iPhone 7 plus,6000.0,مدينتي، القاهرة,/ad/apple-iphone-13-pro-max-ID208943948.html,208943948,legacy,apple,32.0,3.0,NaN,...,False,NaN,phone,phone_brand_detected,1,NaN,NaN,False,4014.808639,1985.191361
1242,سامسونج a50,4000.0,شرم الشيخ، سيناء,/ad/%D8%B3%D8%A7%D9%85%D8%B3%D9%88%D9%86%D8%AC...,208942995,legacy,samsung,128.0,4.0,NaN,...,False,NaN,phone,phone_brand_detected,1,NaN,NaN,False,5040.043991,1040.043991


# MAE by brand

In [ ]:
eval_df['brand_group'] = (eval_df['brand'].fillna('Unknown').astype(str))

#  Calculate MAE statistics for each brand
mae_by_brand = (eval_df
    .groupby('brand_group')
    .agg(
        ads=('absolute_error', 'size'),
        MAE=('absolute_error', 'mean'),
        median_price=('price', 'median')
    )
    .reset_index()
)

mae_by_brand = ( mae_by_brand.sort_values('MAE').rename(columns={'brand_group': 'brand'})) #sorting by mae

display(mae_by_brand.round(2))

,brand,ads,MAE,median_price
0,honor,25,2904.05,14000.0
1,vivo,9,3110.78,14000.0
2,motorola,5,4548.69,12500.0
3,xiaomi,66,4626.41,9000.0
4,infinix,14,4659.81,6950.0
5,oppo,19,4720.37,19300.0
6,huawei,7,5726.10,3000.0
7,samsung,210,6574.21,28000.0
8,realme,15,7074.61,7200.0
9,apple,122,7752.86,25000.0


# MAE by price band

In [13]:
price_bins = [
    -np.inf, 5000, 10000, 20000, 40000, np.inf
]

price_labels = [
    'Budget (<5k)',
    'Lower-mid (5k-10k)',
    'Mid-range (10k-20k)',
    'Upper-mid (20k-40k)',
    'Premium (40k+)'
]


In [ ]:
print("Price summary:")
display(eval_df['price'].describe().round(2))

# 2. Count ads in each proposed price band
eval_df['price_band'] = pd.cut(
    eval_df['price'],
    bins=price_bins,
    labels=price_labels,
    right=False
)

price_band_counts = (eval_df['price_band'].value_counts(sort=False).rename('ads').to_frame())
price_band_counts['percentage'] = (price_band_counts['ads']/ len(eval_df)* 100)
display(price_band_counts.round(2))

Price summary:


count       494.00
mean      24920.32
std       22413.85
min        1000.00
25%        7050.00
50%       17000.00
75%       40000.00
max      149000.00
Name: price, dtype: float64

,ads,percentage
price_band,,
Budget (<5k),70,14.17
Lower-mid (5k-10k),111,22.47
Mid-range (10k-20k),88,17.81
Upper-mid (20k-40k),100,20.24
Premium (40k+),125,25.30


In [14]:
eval_df['price_band'] = pd.cut(
    eval_df['price'],
    bins=price_bins,
    labels=price_labels,
    right=False
)

mae_by_price_band = (
    eval_df
    .groupby('price_band', observed=False)
    .agg(
        ads=('absolute_error', 'size'),
        MAE=('absolute_error', 'mean'),
        median_actual_price=('price', 'median')
    )
    .reset_index()
)

display(mae_by_price_band.round(2))

,price_band,ads,MAE,median_actual_price
0,Budget (<5k),70,2928.02,3150.0
1,Lower-mid (5k-10k),111,5066.13,7300.0
2,Mid-range (10k-20k),88,4238.29,14000.0
3,Upper-mid (20k-40k),100,5975.55,29000.0
4,Premium (40k+),125,10801.84,52000.0


# MAE by model frequency band

In [18]:
def normalize_model_key(series):
    return (series.fillna('unknown').astype(str).str.strip().str.lower())

train_model_data['model_key_norm'] = normalize_model_key(train_model_data['model_key'])

eval_df['model_key_norm'] = normalize_model_key( eval_df['model_key'])


In [23]:
# Count each model in training only
train_model_counts = (train_model_data[ train_model_data['model_key_norm'].ne('unknown')].groupby('model_key_norm').size())

eval_df['train_model_count'] = (eval_df['model_key_norm'].map(train_model_counts).fillna(0).astype(int))


def assign_frequency_band(row):
    key = row['model_key_norm']
    count = row['train_model_count']

    if key == 'unknown':
        return 'Unknown model key'
    elif count == 0:
        return 'Unseen (0)'
    elif count < 5:
        return 'Rare (1-9)'
    elif count < 20:
        return 'Low (10-29)'
    else:
        return 'Frequent (30+)'


eval_df['model_frequency_band'] = eval_df.apply(assign_frequency_band,axis=1)

frequency_order = [
    'Unseen (0)',
    'Rare (1-9)',
    'Low (10-29)',
    'Frequent (30+)',
    'Unknown model key'
]

In [28]:
train_model_counts

model_key_norm
apple iphone 11            237
apple iphone 11 pro         65
apple iphone 11 pro max     56
apple iphone 12             91
apple iphone 12 mini        15
                          ... 
zte nubia z30 pro            3
zte red magic 10 air         1
zte red magic 10 pro         1
zte red magic 7              1
zte unknown                  1
Length: 722, dtype: int64

In [ ]:
mae_by_frequency = (
    eval_df
    .groupby('model_frequency_band', observed=False)
    .agg(
        ads=('absolute_error', 'size'),
        distinct_models=('model_key_norm', 'nunique'),
        MAE=('absolute_error', 'mean'),
        median_price=('price', 'median')
    )
    .reindex(frequency_order)
    .dropna(subset=['ads'])
    .reset_index()
)

display(mae_by_frequency.round(2))

,model_frequency_band,ads,distinct_models,MAE,median_price
0,Unseen (0),48,38,5097.66,9250.0
1,Rare (1-9),113,46,5464.42,10000.0
2,Low (10-29),138,45,6406.56,9100.0
3,Frequent (30+),183,31,6913.01,45000.0
4,Unknown model key,12,1,6391.95,8000.0


In [30]:
unseen_df = eval_df[(eval_df['train_model_count'] == 0) &(eval_df['model_key_norm'] != 'unknown')].copy()
print("Unseen-model ads:", len(unseen_df))

print("Distinct unseen models:",unseen_df['model_key_norm'].nunique())


Unseen-model ads: 48
Distinct unseen models: 38


In [31]:

if len(unseen_df) > 0:
    unseen_mae = mean_absolute_error(unseen_df['price'],unseen_df['predicted_price'])

    unseen_grouped_mae = (unseen_df.assign(
            abs_error=(
                unseen_df['price'] -
                unseen_df['predicted_price']
            ).abs()
        )
        .groupby('title_group')['abs_error']
        .mean()
        .mean()
    )

    print("Unseen-model MAE:", round(unseen_mae, 2))
    print(
        "Unseen-model Grouped MAE:",
        round(unseen_grouped_mae, 2)
    )

    display(
        unseen_df[
            [
                'title', 'brand', 'family', 'model_key',
                'train_model_count', 'price',
                'predicted_price', 'absolute_error'
            ]
        ]
        .sort_values('absolute_error', ascending=False)
        .head(20)
        .round(2)
    )
else:
    print("No unseen known model keys were found in the evaluation set.")

Unseen-model MAE: 5097.66
Unseen-model Grouped MAE: 5226.24


,title,brand,family,model_key,train_model_count,price,predicted_price,absolute_error
575,Huawei Mate X5 ميتx5,huawei,mate,huawei mate x5,0,50000.0,23952.51,26047.49
300,Sony Xperia PRO-I موبايل سوني إكسبيريا برو,sony,xperia,sony xperia pro-i,0,45543.0,24664.84,20878.16
512,إنفنيكس نوت 60,infinix,note,infinix note 60,0,22000.0,7804.56,14195.44
863,شاومي 8 برو ماكس متفتحش زى الفل الشاشه وكله با...,xiaomi,xiaomi,xiaomi 8 pro,0,6000.0,19038.81,13038.81
743,Google Pixel 9 Pro,google,pixel,google pixel 9 pro,0,32000.0,21066.08,10933.92
872,سامسونج a27,samsung,galaxy a,samsung a27,0,18000.0,7689.89,10310.11
261,إنفنيكس نوت 5,infinix,note,infinix note 5,0,12500.0,2252.81,10247.19
319,إنفنيكس نوت 60,infinix,note,infinix note 60,0,16500.0,7992.01,8507.99
698,اوبو ار 15 برو,oppo,r,oppo r15 pro,0,1000.0,8722.23,7722.23
1197,شاومي بوكو اف 6,xiaomi,poco,xiaomi poco f6,0,17000.0,24503.11,7503.11
